In [0]:
import json
import time
import uuid
import calendar
import requests

from datetime import datetime
from delta.tables import DeltaTable

from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, IntegerType, ArrayType
)

from pyspark.sql.functions import (
    col, lit, current_timestamp,
    arrays_zip, explode, to_timestamp
)

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

WEATHER_COLUMNS = [
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
]

RAW_BASE_PATH = (
    "/Volumes/workspace/default/products/"
    "european_climate/raw"
)

BRONZE_TABLE = "bronze_weather"
SILVER_TABLE = "silver_weather"
EXECUTION_LOG_TABLE = "pipeline_execution_logs"
ERROR_LOG_TABLE = "pipeline_error_logs"

FULL_START_YEAR = 1981
FULL_END_YEAR = 2020

CITIES = [
    {"city": "Oslo", "country": "Norway", "region": "Northern Europe",
     "latitude": 59.9139, "longitude": 10.7522},

    {"city": "Stockholm", "country": "Sweden", "region": "Northern Europe",
     "latitude": 59.3293, "longitude": 18.0686},

    {"city": "Helsinki", "country": "Finland", "region": "Northern Europe",
     "latitude": 60.1699, "longitude": 24.9384},

    {"city": "Copenhagen", "country": "Denmark", "region": "Northern Europe",
     "latitude": 55.6761, "longitude": 12.5683},

    {"city": "London", "country": "United Kingdom", "region": "Western Europe",
     "latitude": 51.5074, "longitude": -0.1278},

    {"city": "Berlin", "country": "Germany", "region": "Central Europe",
     "latitude": 52.5200, "longitude": 13.4050},

    {"city": "Madrid", "country": "Spain", "region": "Southern Europe",
     "latitude": 40.4168, "longitude": -3.7038},

    {"city": "Rome", "country": "Italy", "region": "Southern Europe",
     "latitude": 41.9028, "longitude": 12.4964},

    {"city": "Warsaw", "country": "Poland", "region": "Eastern Europe",
     "latitude": 52.2297, "longitude": 21.0122},
]

NORMAL_DELAY_SECONDS = 6.0
MAX_RETRIES = 8

TOTAL_BATCHES = len(CITIES) * (FULL_END_YEAR - FULL_START_YEAR + 1)

print("Cities:", len(CITIES))
print("Years:", FULL_START_YEAR, "→", FULL_END_YEAR)
print("Expected city-year batches:", TOTAL_BATCHES)
print("Weather variables:", len(WEATHER_COLUMNS))
print("API pacing:", NORMAL_DELAY_SECONDS, "seconds")


Cities: 9
Years: 1981 → 2020
Expected city-year batches: 360
Weather variables: 12
API pacing: 6.0 seconds


In [0]:
hourly_units_schema = StructType([
    StructField("time", StringType(), True),
    *[StructField(name, StringType(), True) for name in WEATHER_COLUMNS],
])

hourly_schema = StructType([
    StructField("time", ArrayType(StringType()), True),
    StructField("temperature_2m", ArrayType(DoubleType()), True),
    StructField("relative_humidity_2m", ArrayType(DoubleType()), True),
    StructField("apparent_temperature", ArrayType(DoubleType()), True),
    StructField("precipitation", ArrayType(DoubleType()), True),
    StructField("rain", ArrayType(DoubleType()), True),
    StructField("snowfall", ArrayType(DoubleType()), True),
    StructField("weather_code", ArrayType(IntegerType()), True),
    StructField("pressure_msl", ArrayType(DoubleType()), True),
    StructField("surface_pressure", ArrayType(DoubleType()), True),
    StructField("cloud_cover", ArrayType(DoubleType()), True),
    StructField("wind_speed_10m", ArrayType(DoubleType()), True),
    StructField("wind_direction_10m", ArrayType(DoubleType()), True),
])

raw_weather_schema = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("generationtime_ms", DoubleType(), True),
    StructField("utc_offset_seconds", IntegerType(), True),
    StructField("timezone", StringType(), True),
    StructField("timezone_abbreviation", StringType(), True),
    StructField("elevation", DoubleType(), True),
    StructField("hourly_units", hourly_units_schema, True),
    StructField("hourly", hourly_schema, True),
])

def make_batch_id(city_info, start_date, end_date):
    safe_city = city_info["city"].lower().replace(" ", "_")
    return f"{safe_city}_{start_date}_{end_date}"

def raw_to_bronze(city_info, file_path, start_date, end_date, load_type):
    batch_id = make_batch_id(city_info, start_date, end_date)

    raw_df = (
        spark.read
        .schema(raw_weather_schema)
        .option("multiLine", "true")
        .json(file_path)
    )

    return (
        raw_df
        .withColumn("city", lit(city_info["city"]))
        .withColumn("country", lit(city_info["country"]))
        .withColumn("region", lit(city_info["region"]))
        .withColumn("source", lit("Open-Meteo ERA5"))
        .withColumn("batch_id", lit(batch_id))
        .withColumn("load_type", lit(load_type))
        .withColumn("raw_file_path", lit(file_path))
        .withColumn("load_timestamp", current_timestamp())
    )

def bronze_to_silver(bronze_df):
    exploded_df = bronze_df.select(
        "city",
        "country",
        "region",
        "source",
        "batch_id",
        "load_type",
        "raw_file_path",
        col("load_timestamp").alias("bronze_load_timestamp"),
        explode(
            arrays_zip(
                col("hourly.time"),
                col("hourly.temperature_2m"),
                col("hourly.relative_humidity_2m"),
                col("hourly.apparent_temperature"),
                col("hourly.precipitation"),
                col("hourly.rain"),
                col("hourly.snowfall"),
                col("hourly.weather_code"),
                col("hourly.pressure_msl"),
                col("hourly.surface_pressure"),
                col("hourly.cloud_cover"),
                col("hourly.wind_speed_10m"),
                col("hourly.wind_direction_10m"),
            )
        ).alias("weather"),
    )

    return exploded_df.select(
        "city",
        "country",
        "region",
        to_timestamp(col("weather.time"), "yyyy-MM-dd'T'HH:mm").alias("timestamp"),
        col("weather.temperature_2m").cast("double").alias("temperature_2m"),
        col("weather.relative_humidity_2m").cast("double").alias("relative_humidity_2m"),
        col("weather.apparent_temperature").cast("double").alias("apparent_temperature"),
        col("weather.precipitation").cast("double").alias("precipitation"),
        col("weather.rain").cast("double").alias("rain"),
        col("weather.snowfall").cast("double").alias("snowfall"),
        col("weather.weather_code").cast("int").alias("weather_code"),
        col("weather.pressure_msl").cast("double").alias("pressure_msl"),
        col("weather.surface_pressure").cast("double").alias("surface_pressure"),
        col("weather.cloud_cover").cast("double").alias("cloud_cover"),
        col("weather.wind_speed_10m").cast("double").alias("wind_speed_10m"),
        col("weather.wind_direction_10m").cast("double").alias("wind_direction_10m"),
        "source",
        "batch_id",
        "load_type",
        "raw_file_path",
        "bronze_load_timestamp",
        current_timestamp().alias("load_timestamp"),
    )

print("Explicit schema and transformations ready.")


Explicit schema and transformations ready.


In [0]:
EXPECTED_TOP_LEVEL_KEYS = {
    "latitude", "longitude", "generationtime_ms",
    "utc_offset_seconds", "timezone",
    "timezone_abbreviation", "elevation",
    "hourly_units", "hourly"
}

EXPECTED_HOURLY_KEYS = set(WEATHER_COLUMNS) | {"time"}

def build_raw_file_path(city_info, start_date, end_date, load_type="FULL"):
    safe_city = city_info["city"].lower().replace(" ", "_")
    folder = f"{RAW_BASE_PATH}/{load_type.lower()}"
    return f"{folder}/{safe_city}_{start_date}_{end_date}.json"

def raw_file_exists(file_path):
    try:
        dbutils.fs.head(file_path, 1)
        return True
    except Exception:
        return False

def read_raw_json(file_path):
    text = dbutils.fs.head(file_path, 100_000_000)
    return json.loads(text)

def validate_raw_file(file_path):
    reasons = []

    try:
        data = read_raw_json(file_path)
    except Exception as e:
        return False, [f"Invalid JSON: {e}"]

    missing_top = EXPECTED_TOP_LEVEL_KEYS - set(data.keys())
    if missing_top:
        reasons.append(
            "Missing top-level key(s): " + ", ".join(sorted(missing_top))
        )

    hourly = data.get("hourly")
    if not isinstance(hourly, dict):
        return False, reasons + ["hourly is not a valid object"]

    missing_hourly = EXPECTED_HOURLY_KEYS - set(hourly.keys())
    if missing_hourly:
        reasons.append(
            "Missing hourly key(s): " + ", ".join(sorted(missing_hourly))
        )

    if isinstance(hourly.get("time"), list):
        expected_length = len(hourly["time"])

        if expected_length == 0:
            reasons.append("hourly.time is empty")

        for column in EXPECTED_HOURLY_KEYS:
            if column in hourly:
                if not isinstance(hourly[column], list):
                    reasons.append(f"hourly.{column} is not an array")
                elif len(hourly[column]) != expected_length:
                    reasons.append(f"hourly.{column} array length mismatch")
    else:
        reasons.append("hourly.time is not an array")

    return len(reasons) == 0, reasons

def fetch_weather(city_info, start_date, end_date, max_retries=MAX_RETRIES):
    params = {
        "latitude": city_info["latitude"],
        "longitude": city_info["longitude"],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(WEATHER_COLUMNS),
        "timezone": "UTC",
        "models": "era5",
    }

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                BASE_URL,
                params=params,
                timeout=180,
            )

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")

                if retry_after and retry_after.isdigit():
                    wait_seconds = int(retry_after)
                else:
                    wait_seconds = min(30 * (2 ** (attempt - 1)), 300)

                print(
                    f"429 Too Many Requests → "
                    f"retry {attempt}/{max_retries} in {wait_seconds}s"
                )
                time.sleep(wait_seconds)
                continue

            if response.status_code in (500, 502, 503, 504):
                wait_seconds = min(10 * attempt, 60)
                print(
                    f"Server error {response.status_code} → "
                    f"retry {attempt}/{max_retries} in {wait_seconds}s"
                )
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            response.json()
            return response

        except (requests.RequestException, ValueError):
            if attempt == max_retries:
                raise

            wait_seconds = min(10 * attempt, 60)
            print(
                f"Request/JSON error → "
                f"retry {attempt}/{max_retries} in {wait_seconds}s"
            )
            time.sleep(wait_seconds)

    raise RuntimeError("API request failed after all retries.")

def save_raw_response(response, city_info, start_date, end_date, load_type):
    file_path = build_raw_file_path(
        city_info, start_date, end_date, load_type
    )

    folder = file_path.rsplit("/", 1)[0]
    dbutils.fs.mkdirs(folder)

    dbutils.fs.put(
        file_path,
        response.text,
        overwrite=True
    )

    return file_path

def ensure_valid_raw_file(city_info, start_date, end_date, load_type):
    file_path = build_raw_file_path(
        city_info, start_date, end_date, load_type
    )

    if raw_file_exists(file_path):
        valid, reasons = validate_raw_file(file_path)

        if valid:
            print("RAW      → VALID EXISTING FILE REUSED")
            return file_path, "REUSED"

        print("RAW      → EXISTING FILE INVALID")
        print("Reason:", " | ".join(reasons))
        print("RAW      → RE-DOWNLOADING")

    response = fetch_weather(
        city_info,
        start_date,
        end_date
    )

    file_path = save_raw_response(
        response,
        city_info,
        start_date,
        end_date,
        load_type
    )

    valid, reasons = validate_raw_file(file_path)

    if not valid:
        raise ValueError(
            "Downloaded raw file failed validation: "
            + " | ".join(reasons)
        )

    print(
        "RAW      → DOWNLOADED",
        f"{len(response.content) / (1024 ** 2):.3f} MB"
    )

    return file_path, "DOWNLOADED"

def write_execution_log(
    run_id,
    batch_id,
    layer,
    load_type,
    parameter_or_file,
    start_time,
    end_time,
    status,
    rows_inserted=0,
    rows_updated=0,
    error_message=None
):
    record = [{
        "run_id": run_id,
        "batch_id": batch_id,
        "layer": layer,
        "load_type": load_type,
        "parameter_or_file": parameter_or_file,
        "start_time": start_time,
        "end_time": end_time,
        "status": status,
        "rows_inserted": int(rows_inserted),
        "rows_updated": int(rows_updated),
        "error_message": error_message,
    }]

    (
        spark.createDataFrame(
            record,
            schema=spark.table(EXECUTION_LOG_TABLE).schema
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(EXECUTION_LOG_TABLE)
    )

def write_error_log(
    run_id,
    batch_id,
    layer,
    source_file,
    error
):
    record = [{
        "error_id": str(uuid.uuid4()),
        "run_id": run_id,
        "batch_id": batch_id,
        "layer": layer,
        "source_file": source_file,
        "error_type": type(error).__name__,
        "error_message": str(error),
        "error_timestamp": datetime.now(),
    }]

    (
        spark.createDataFrame(
            record,
            schema=spark.table(ERROR_LOG_TABLE).schema
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(ERROR_LOG_TABLE)
    )

print("API/raw/logging helpers ready.")


API/raw/logging helpers ready.


In [0]:
def merge_bronze(bronze_df):
    target = DeltaTable.forName(spark, BRONZE_TABLE)

    (
        target.alias("target")
        .merge(
            bronze_df.alias("source"),
            "target.batch_id = source.batch_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    metrics = (
        target.history(1)
        .select("operationMetrics")
        .first()["operationMetrics"]
    )

    return (
        int(metrics.get("numTargetRowsInserted", 0)),
        int(metrics.get("numTargetRowsUpdated", 0))
    )

def merge_silver(silver_df):
    target = DeltaTable.forName(spark, SILVER_TABLE)

    change_condition = "NOT (" + " AND ".join(
        [
            f"target.{column} <=> source.{column}"
            for column in WEATHER_COLUMNS
        ]
    ) + ")"

    update_values = {
        column: f"source.{column}"
        for column in WEATHER_COLUMNS
    }

    update_values.update({
        "country": "source.country",
        "region": "source.region",
        "source": "source.source",
        "batch_id": "source.batch_id",
        "load_type": "source.load_type",
        "raw_file_path": "source.raw_file_path",
        "bronze_load_timestamp": "source.bronze_load_timestamp",
        "load_timestamp": "current_timestamp()",
    })

    (
        target.alias("target")
        .merge(
            silver_df.alias("source"),
            "target.city = source.city AND target.timestamp = source.timestamp"
        )
        .whenMatchedUpdate(
            condition=change_condition,
            set=update_values
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    metrics = (
        target.history(1)
        .select("operationMetrics")
        .first()["operationMetrics"]
    )

    return (
        int(metrics.get("numTargetRowsInserted", 0)),
        int(metrics.get("numTargetRowsUpdated", 0))
    )

def batch_logged_successfully(batch_id, load_type="FULL"):
    return (
        spark.table(EXECUTION_LOG_TABLE)
        .filter(
            (col("batch_id") == batch_id)
            & (col("load_type") == load_type)
            & (col("layer") == "Silver")
            & (col("status") == "SUCCESS")
        )
        .limit(1)
        .count()
        > 0
    )

def batch_exists_in_silver(batch_id):
    return (
        spark.table(SILVER_TABLE)
        .filter(col("batch_id") == batch_id)
        .limit(1)
        .count()
        > 0
    )

def batch_already_completed(batch_id, load_type="FULL"):
    return (
        batch_logged_successfully(batch_id, load_type)
        and batch_exists_in_silver(batch_id)
    )

def run_one_batch(
    city_info,
    start_date,
    end_date,
    load_type="FULL",
    skip_completed=True
):
    run_id = str(uuid.uuid4())
    batch_id = make_batch_id(city_info, start_date, end_date)
    pipeline_start = datetime.now()

    print("\n======================================")
    print(city_info["city"], start_date, "→", end_date)
    print("Batch:", batch_id)

    if skip_completed and batch_already_completed(batch_id, load_type):
        print("FINAL    → SKIPPED (already present in Silver)")
        return {"batch_id": batch_id, "status": "SKIPPED"}

    file_path = build_raw_file_path(
        city_info, start_date, end_date, load_type
    )

    try:
        raw_start = datetime.now()

        file_path, raw_source = ensure_valid_raw_file(
            city_info,
            start_date,
            end_date,
            load_type
        )

        raw_end = datetime.now()

        write_execution_log(
            run_id, batch_id, "Raw", load_type,
            file_path, raw_start, raw_end, "SUCCESS"
        )

        bronze_start = datetime.now()

        bronze_df = raw_to_bronze(
            city_info, file_path,
            start_date, end_date, load_type
        )

        bronze_inserted, bronze_updated = merge_bronze(bronze_df)

        bronze_end = datetime.now()

        write_execution_log(
            run_id, batch_id, "Bronze", load_type,
            file_path, bronze_start, bronze_end, "SUCCESS",
            bronze_inserted, bronze_updated
        )

        print(
            "BRONZE   → SUCCESS",
            "| inserted:", bronze_inserted,
            "| updated:", bronze_updated
        )

        silver_start = datetime.now()

        silver_df = (
            bronze_to_silver(bronze_df)
            .dropDuplicates(["city", "timestamp"])
        )

        null_timestamps = (
            silver_df
            .filter(col("timestamp").isNull())
            .count()
        )

        if null_timestamps > 0:
            raise ValueError(
                f"Silver batch contains {null_timestamps} null timestamp(s)"
            )

        silver_inserted, silver_updated = merge_silver(silver_df)

        silver_end = datetime.now()

        write_execution_log(
            run_id, batch_id, "Silver", load_type,
            file_path, silver_start, silver_end, "SUCCESS",
            silver_inserted, silver_updated
        )

        print(
            "SILVER   → SUCCESS",
            "| inserted:", silver_inserted,
            "| updated:", silver_updated
        )

        print("FINAL    → SUCCESS")

        return {
            "batch_id": batch_id,
            "status": "SUCCESS",
            "raw_source": raw_source,
            "bronze_inserted": bronze_inserted,
            "bronze_updated": bronze_updated,
            "silver_inserted": silver_inserted,
            "silver_updated": silver_updated
        }

    except Exception as e:
        failure_time = datetime.now()

        print("FINAL    → FAILURE")
        print("Reason:", str(e))

        write_execution_log(
            run_id, batch_id, "Pipeline", load_type,
            file_path, pipeline_start, failure_time, "FAILURE",
            0, 0, str(e)
        )

        write_error_log(
            run_id, batch_id, "Pipeline", file_path, e
        )

        return {
            "batch_id": batch_id,
            "status": "FAILURE",
            "error": str(e)
        }

print("Production MERGE + batch runner ready.")


Production MERGE + batch runner ready.


In [0]:
already_complete = 0
need_processing = 0

for city_info in CITIES:
    for year in range(FULL_START_YEAR, FULL_END_YEAR + 1):
        start_date = f"{year}-01-01"
        end_date = f"{year}-12-31"

        batch_id = make_batch_id(
            city_info,
            start_date,
            end_date
        )

        if batch_already_completed(batch_id, "FULL"):
            already_complete += 1
        else:
            need_processing += 1

print("===== PRE-FLIGHT =====")
print("Expected batches:", TOTAL_BATCHES)
print("Already complete AND present in Silver:", already_complete)
print("Need processing/reprocessing:", need_processing)
print(
    "Some missing Silver batches may still reuse valid raw JSON files, "
    "so they do not necessarily require a new API request."
)


===== PRE-FLIGHT =====
Expected batches: 360
Already complete AND present in Silver: 135
Need processing/reprocessing: 225
Some missing Silver batches may still reuse valid raw JSON files, so they do not necessarily require a new API request.


In [0]:
full_success = 0
full_failed = 0
full_skipped = 0
failed_batches = []

batch_number = 0

print("======================================")
print("FULL PRODUCTION LOAD")
print("Cities:", len(CITIES))
print("Years:", FULL_START_YEAR, "→", FULL_END_YEAR)
print("Expected batches:", TOTAL_BATCHES)
print("======================================")

for city_info in CITIES:
    for year in range(FULL_START_YEAR, FULL_END_YEAR + 1):
        batch_number += 1

        start_date = f"{year}-01-01"
        end_date = f"{year}-12-31"

        print(f"\nBatch {batch_number}/{TOTAL_BATCHES}")

        result = run_one_batch(
            city_info=city_info,
            start_date=start_date,
            end_date=end_date,
            load_type="FULL",
            skip_completed=True
        )

        if result["status"] == "SUCCESS":
            full_success += 1

        elif result["status"] == "SKIPPED":
            full_skipped += 1

        else:
            full_failed += 1
            failed_batches.append({
                "city": city_info["city"],
                "year": year,
                "batch_id": result["batch_id"],
                "error": result.get("error", "Unknown error")
            })

        print(
            "Progress:",
            batch_number, "/", TOTAL_BATCHES,
            "| Success:", full_success,
            "| Skipped:", full_skipped,
            "| Failed:", full_failed
        )

        time.sleep(NORMAL_DELAY_SECONDS)

print("\n======================================")
print("FULL LOAD PASS FINISHED")
print("Success:", full_success)
print("Skipped:", full_skipped)
print("Failed:", full_failed)
print("======================================")

if failed_batches:
    print("\n===== FAILED BATCHES =====")
    for item in failed_batches:
        print(
            item["city"],
            item["year"],
            "→",
            item["error"]
        )
else:
    print("\nNo failed batches.")


FULL PRODUCTION LOAD
Cities: 9
Years: 1981 → 2020
Expected batches: 360

Batch 1/360

Oslo 1981-01-01 → 1981-12-31
Batch: oslo_1981-01-01_1981-12-31
[Truncated to first 1 bytes]
RAW      → VALID EXISTING FILE REUSED
BRONZE   → SUCCESS | inserted: 1 | updated: 0
SILVER   → SUCCESS | inserted: 8760 | updated: 0
FINAL    → SUCCESS
Progress: 1 / 360 | Success: 1 | Skipped: 0 | Failed: 0

Batch 2/360

Oslo 1982-01-01 → 1982-12-31
Batch: oslo_1982-01-01_1982-12-31
[Truncated to first 1 bytes]
RAW      → VALID EXISTING FILE REUSED
BRONZE   → SUCCESS | inserted: 1 | updated: 0
SILVER   → SUCCESS | inserted: 8760 | updated: 0
FINAL    → SUCCESS
Progress: 2 / 360 | Success: 2 | Skipped: 0 | Failed: 0

Batch 3/360

Oslo 1983-01-01 → 1983-12-31
Batch: oslo_1983-01-01_1983-12-31
[Truncated to first 1 bytes]
RAW      → VALID EXISTING FILE REUSED
BRONZE   → SUCCESS | inserted: 1 | updated: 0
SILVER   → SUCCESS | inserted: 8760 | updated: 0
FINAL    → SUCCESS
Progress: 3 / 360 | Success: 3 | Skipped: 

In [0]:
selected_city_names = [c["city"] for c in CITIES]

scope_silver = (
    spark.table(SILVER_TABLE)
    .filter(col("city").isin(selected_city_names))
    .filter(
        (col("timestamp") >= f"{FULL_START_YEAR}-01-01")
        & (col("timestamp") < f"{FULL_END_YEAR + 1}-01-01")
    )
)

total_rows = scope_silver.count()

distinct_keys = (
    scope_silver
    .select("city", "timestamp")
    .distinct()
    .count()
)

duplicates = total_rows - distinct_keys

city_count = (
    scope_silver
    .select("city")
    .distinct()
    .count()
)

null_timestamps = (
    scope_silver
    .filter(col("timestamp").isNull())
    .count()
)

null_load_timestamps = (
    scope_silver
    .filter(col("load_timestamp").isNull())
    .count()
)

date_range = (
    scope_silver
    .selectExpr(
        "MIN(timestamp) AS min_timestamp",
        "MAX(timestamp) AS max_timestamp"
    )
    .first()
)

successful_scope_batches = 0
missing_scope_batches = []

for city_info in CITIES:
    for year in range(FULL_START_YEAR, FULL_END_YEAR + 1):
        start_date = f"{year}-01-01"
        end_date = f"{year}-12-31"

        batch_id = make_batch_id(
            city_info,
            start_date,
            end_date
        )

        if batch_already_completed(batch_id, "FULL"):
            successful_scope_batches += 1
        else:
            missing_scope_batches.append(batch_id)

expected_days_per_city = sum(
    366 if calendar.isleap(year) else 365
    for year in range(FULL_START_YEAR, FULL_END_YEAR + 1)
)

expected_rows = len(CITIES) * expected_days_per_city * 24

print("===== FINAL FULL-LOAD VALIDATION =====")
print("Silver rows in final scope:", total_rows)
print("Expected hourly rows:", expected_rows)
print("Distinct city + timestamp:", distinct_keys)
print("Duplicates:", duplicates)
print("Cities:", city_count)
print("Null timestamps:", null_timestamps)
print("Null load_timestamp:", null_load_timestamps)
print("Earliest timestamp:", date_range["min_timestamp"])
print("Latest timestamp:", date_range["max_timestamp"])
print("Completed scope batches:", successful_scope_batches, "/", TOTAL_BATCHES)
print("Missing scope batches:", len(missing_scope_batches))

checks = {
    "9 cities present":
        city_count == len(CITIES),

    "All 360 batches present":
        successful_scope_batches == TOTAL_BATCHES,

    "Expected hourly row count":
        total_rows == expected_rows,

    "No duplicate city + timestamp":
        duplicates == 0,

    "No null timestamps":
        null_timestamps == 0,

    "No null load_timestamp":
        null_load_timestamps == 0,

    "Starts in 1981": (
        date_range["min_timestamp"] is not None
        and date_range["min_timestamp"].year == FULL_START_YEAR
    ),

    "Reaches 2020": (
        date_range["max_timestamp"] is not None
        and date_range["max_timestamp"].year == FULL_END_YEAR
    ),
}

print("\n===== FINAL CHECKLIST =====")
for name, passed in checks.items():
    print(name, "→", "PASS" if passed else "FAIL")

if missing_scope_batches:
    print("\nFirst missing batches:")
    for batch_id in missing_scope_batches[:20]:
        print(batch_id)

if all(checks.values()):
    print("\nFINAL RESULT: FULL LOAD VALIDATION PASSED")
else:
    print("\nFINAL RESULT: CHECK FAILED ITEMS / RETRY MISSING BATCHES")


===== FINAL FULL-LOAD VALIDATION =====
Silver rows in final scope: 3155760
Expected hourly rows: 3155760
Distinct city + timestamp: 3155760
Duplicates: 0
Cities: 9
Null timestamps: 0
Null load_timestamp: 0
Earliest timestamp: 1981-01-01 00:00:00
Latest timestamp: 2020-12-31 23:00:00
Completed scope batches: 360 / 360
Missing scope batches: 0

===== FINAL CHECKLIST =====
9 cities present → PASS
All 360 batches present → PASS
Expected hourly row count → PASS
No duplicate city + timestamp → PASS
No null timestamps → PASS
No null load_timestamp → PASS
Starts in 1981 → PASS
Reaches 2020 → PASS

FINAL RESULT: FULL LOAD VALIDATION PASSED


In [0]:
files = dbutils.fs.ls(
    "/Volumes/workspace/default/products/european_climate/raw/full/"
)

total_bytes = sum(f.size for f in files)

total_mb = total_bytes / (1024 ** 2)
total_gb = total_bytes / (1024 ** 3)

print("Raw files:", len(files))
print(f"Total raw size: {total_mb:.2f} MB")
print(f"Total raw size: {total_gb:.3f} GB")

Raw files: 565
Total raw size: 347.05 MB
Total raw size: 0.339 GB


In [0]:
final_cities = [
    "oslo",
    "stockholm",
    "helsinki",
    "copenhagen",
    "london",
    "berlin",
    "madrid",
    "rome",
    "warsaw"
]

files = dbutils.fs.ls(
    "/Volumes/workspace/default/products/european_climate/raw/full/"
)

final_files = []

for f in files:
    name = f.name.lower()

    city_match = any(
        name.startswith(city + "_")
        for city in final_cities
    )

    year_match = any(
        f"_{year}-01-01_" in name
        for year in range(1980, 2021)
    )

    if city_match and year_match:
        final_files.append(f)

total_bytes = sum(f.size for f in final_files)

print("Final-scope raw files:", len(final_files))
print(f"Final-scope raw size: {total_bytes / (1024**2):.2f} MB")

Final-scope raw files: 365
Final-scope raw size: 224.62 MB


In [0]:
bronze_detail = spark.sql("""
DESCRIBE DETAIL bronze_weather
""")

bronze_detail.select(
    "numFiles",
    "sizeInBytes"
).show(truncate=False)

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|1       |11810317   |
+--------+-----------+



In [0]:
bronze_size_bytes = bronze_detail.select("sizeInBytes").first()[0]

bronze_size_mb = bronze_size_bytes / (1024 ** 2)
bronze_size_gb = bronze_size_bytes / (1024 ** 3)

print(f"Bronze size: {bronze_size_mb:.2f} MB")
print(f"Bronze size: {bronze_size_gb:.3f} GB")

Bronze size: 11.26 MB
Bronze size: 0.011 GB


In [0]:
silver_detail = spark.sql("""
DESCRIBE DETAIL silver_weather
""")

silver_size_bytes = silver_detail.select("sizeInBytes").first()[0]

silver_size_mb = silver_size_bytes / (1024 ** 2)
silver_size_gb = silver_size_bytes / (1024 ** 3)

print(f"Silver size: {silver_size_mb:.2f} MB")
print(f"Silver size: {silver_size_gb:.3f} GB")

Silver size: 35.59 MB
Silver size: 0.035 GB


In [0]:
silver_rows = spark.table("silver_weather").count()

print("Silver total rows:", silver_rows)

Silver total rows: 3155760


In [0]:
silver_df = spark.table("silver_weather")

display(silver_df.limit(20))

city,country,region,timestamp,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,weather_code,pressure_msl,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,source,batch_id,load_type,raw_file_path,bronze_load_timestamp,load_timestamp
Berlin,Germany,Central Europe,2023-01-01T15:00:00.000Z,13.9,65.0,10.9,0.0,0.0,0.0,3,1017.1,1012.6,100.0,16.7,237.0,Open-Meteo ERA5,incremental_berlin_2023-01-01_2023-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/berlin_2023-01-01_2023-12-31.json,2026-10-06T20:47:56.640Z,2026-10-06T20:47:56.640Z
Berlin,Germany,Central Europe,2023-01-02T10:00:00.000Z,14.1,66.0,11.0,0.2,0.2,0.0,51,1014.2,1009.7,80.0,17.5,217.0,Open-Meteo ERA5,incremental_berlin_2023-01-01_2023-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/berlin_2023-01-01_2023-12-31.json,2026-10-06T20:47:56.640Z,2026-10-06T20:47:56.640Z
Berlin,Germany,Central Europe,2023-01-02T16:00:00.000Z,13.2,68.0,9.7,0.0,0.0,0.0,3,1015.9,1011.4,100.0,19.4,255.0,Open-Meteo ERA5,incremental_berlin_2023-01-01_2023-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/berlin_2023-01-01_2023-12-31.json,2026-10-06T20:47:56.640Z,2026-10-06T20:47:56.640Z
Berlin,Germany,Central Europe,2023-01-02T22:00:00.000Z,9.0,84.0,6.0,0.3,0.3,0.0,51,1020.6,1016.0,99.0,14.8,267.0,Open-Meteo ERA5,incremental_berlin_2023-01-01_2023-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/berlin_2023-01-01_2023-12-31.json,2026-10-06T20:47:56.640Z,2026-10-06T20:47:56.640Z
Berlin,Germany,Central Europe,2023-01-03T10:00:00.000Z,6.0,83.0,2.3,0.0,0.0,0.0,2,1028.1,1023.5,77.0,14.8,274.0,Open-Meteo ERA5,incremental_berlin_2023-01-01_2023-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/berlin_2023-01-01_2023-12-31.json,2026-10-06T20:47:56.640Z,2026-10-06T20:47:56.640Z
Berlin,Germany,Central Europe,2023-01-03T16:00:00.000Z,4.0,85.0,1.2,0.0,0.0,0.0,1,1028.7,1024.0,47.0,6.6,241.0,Open-Meteo ERA5,incremental_berlin_2023-01-01_2023-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/berlin_2023-01-01_2023-12-31.json,2026-10-06T20:47:56.640Z,2026-10-06T20:47:56.640Z
Berlin,Germany,Central Europe,2023-01-04T02:00:00.000Z,3.2,90.0,-0.9,0.0,0.0,0.0,1,1024.9,1020.2,46.0,15.6,195.0,Open-Meteo ERA5,incremental_berlin_2023-01-01_2023-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/berlin_2023-01-01_2023-12-31.json,2026-10-06T20:47:56.640Z,2026-10-06T20:47:56.640Z
Berlin,Germany,Central Europe,2023-01-05T04:00:00.000Z,9.6,82.0,4.4,0.1,0.1,0.0,51,1007.3,1002.8,86.0,29.5,269.0,Open-Meteo ERA5,incremental_berlin_2023-01-01_2023-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/berlin_2023-01-01_2023-12-31.json,2026-10-06T20:47:56.640Z,2026-10-06T20:47:56.640Z
Berlin,Germany,Central Europe,2023-01-05T20:00:00.000Z,4.6,78.0,0.8,0.0,0.0,0.0,3,1020.2,1015.6,91.0,12.4,324.0,Open-Meteo ERA5,incremental_berlin_2023-01-01_2023-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/berlin_2023-01-01_2023-12-31.json,2026-10-06T20:47:56.640Z,2026-10-06T20:47:56.640Z
Berlin,Germany,Central Europe,2023-01-06T01:00:00.000Z,2.2,90.0,-0.5,0.0,0.0,0.0,3,1021.4,1016.7,100.0,5.1,135.0,Open-Meteo ERA5,incremental_berlin_2023-01-01_2023-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/berlin_2023-01-01_2023-12-31.json,2026-10-06T20:47:56.640Z,2026-10-06T20:47:56.640Z


In [0]:
from pyspark.sql.functions import col, countDistinct, min, max

silver = spark.table("silver_weather")

full_data = silver.filter(
    (col("timestamp") >= "1981-01-01") &
    (col("timestamp") < "2021-01-01")
)

total_rows = full_data.count()

distinct_keys = (
    full_data.select("city", "timestamp")
    .distinct()
    .count()
)

print("===== FULL LOAD VALIDATION =====")

print("Total rows:", total_rows)
print("Duplicate keys:", total_rows - distinct_keys)

full_data.agg(
    countDistinct("city").alias("total_cities"),
    min("timestamp").alias("start_date"),
    max("timestamp").alias("end_date")
).show()

display(
    full_data.groupBy("city")
    .count()
    .orderBy("city")
)

assert total_rows == 3155760
assert total_rows == distinct_keys

print("FULL LOAD VALIDATION PASSED")

===== FULL LOAD VALIDATION =====
Total rows: 3155760
Duplicate keys: 0
+------------+-------------------+-------------------+
|total_cities|         start_date|           end_date|
+------------+-------------------+-------------------+
|           9|1981-01-01 00:00:00|2020-12-31 23:00:00|
+------------+-------------------+-------------------+



city,count
Berlin,350640
Copenhagen,350640
Helsinki,350640
London,350640
Madrid,350640
Oslo,350640
Rome,350640
Stockholm,350640
Warsaw,350640


FULL LOAD VALIDATION PASSED
